In [8]:
# 필요 패키지 (최초 1회)
# 주석 해제 후 실행하세요.
# %pip install ucimlrepo kaggle pandas pyarrow requests tqdm
# %pip install ucimlrepo certifi

  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)

   -------------------- ------------------- 1/2 [ucimlrepo]
   ---------------------------------------- 2/2 [ucimlrepo]

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import sys, subprocess, shutil, os

# 프로젝트 루트 — 노트북이 notebooks/ 안에 있다고 가정하고 상위로.
# 루트에서 직접 실행 중이면 Path.cwd() 로 잡히도록 처리.
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]

DATA      = ROOT / "data"
RAW       = DATA / "raw"
PROCESSED = DATA / "processed"
EXTERNAL  = DATA / "external"

for p in [RAW, PROCESSED, EXTERNAL,
          ROOT / "src", ROOT / "configs",
          ROOT / "artifacts" / "models",
          ROOT / "artifacts" / "figures",
          ROOT / "artifacts" / "tables"]:
    p.mkdir(parents=True, exist_ok=True)

print("프로젝트 루트:", ROOT)
print("raw       :", RAW)
print("processed :", PROCESSED)


프로젝트 루트: C:\Users\User\Downloads\학술\30_연구기획4개
raw       : C:\Users\User\Downloads\학술\30_연구기획4개\data\raw
processed : C:\Users\User\Downloads\학술\30_연구기획4개\data\processed


In [5]:
# .gitignore — 원본 데이터는 git에서 제외 (용량·라이선스)
gitignore = ROOT / ".gitignore"
rules = [
    "data/raw/", "data/processed/", "data/external/",
    "artifacts/models/", "*.parquet", "__pycache__/",
    ".ipynb_checkpoints/", ".env",
]
existing = gitignore.read_text().splitlines() if gitignore.exists() else []
new = [r for r in rules if r not in existing]
if new:
    with gitignore.open("a") as f:
        if existing: f.write("\n")
        f.write("\n".join(new) + "\n")
    print("추가된 .gitignore 규칙:", new)
else:
    print(".gitignore 이미 최신 상태")


.gitignore 이미 최신 상태


In [6]:
import pandas as pd

def report(df: pd.DataFrame, name: str, label_col: str | None = None):
    """공통 검증 리포트: 행·열·결측·라벨분포."""
    print(f"── {name} ──")
    print(f"  shape        : {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    miss = df.isna().mean()
    print(f"  결측 평균    : {miss.mean():.3%}  (최대 {miss.max():.3%})")
    if label_col and label_col in df.columns:
        vc = df[label_col].value_counts(dropna=False)
        print(f"  라벨 분포    :")
        for k, v in vc.items():
            print(f"      {k}: {v:,} ({v/len(df):.2%})")
    print()

def save_processed(df: pd.DataFrame, filename: str):
    out = PROCESSED / filename
    df.to_parquet(out, index=False)
    print(f"  → 저장: {out.relative_to(ROOT)}  ({out.stat().st_size/1e6:.1f} MB)")
    return out


---
## 1. Polish Companies Bankruptcy (UCI #365) — 완전자동

`ucimlrepo`로 바로 fetch. 5개 예측기간(1stYear~5thYear)이 개별 케이스로 제공됩니다.
UCI 패키지는 이들을 하나로 합쳐 `year` 구분 없이 주므로, 여기서는 원본 그대로 저장합니다.
라이선스: **CC BY 4.0** (출처 표기 조건).

In [7]:
def fetch_polish():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=365)
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    return df.rename(columns={ds.data.targets.columns[0]: "bankrupt"})

try:
    polish = fetch_polish()
except Exception as e:
    print("⚠️ 1차 시도 실패:", type(e).__name__, "-", e)
    print("   SSL 인증서 오류라면 아래 폴백을 시도합니다...")
    try:
        import ssl
        # 검증 우회는 최후 수단 — 신뢰 가능한 네트워크에서만 사용하세요.
        ssl._create_default_https_context = ssl._create_unverified_context
        polish = fetch_polish()
        print("   폴백 성공 (SSL 검증 우회).")
    except Exception as e2:
        polish = None
        print("⚠️ Polish 수집 최종 실패:", type(e2).__name__, "-", e2)
        print("   대안: UCI 페이지에서 zip 수동 다운로드 → data/raw/polish/ 배치")

if polish is not None:
    report(polish, "Polish (UCI #365)", label_col="bankrupt")
    save_processed(polish, "polish.parquet")

── Polish (UCI #365) ──
  shape        : 43,405 rows × 66 cols
  결측 평균    : 1.442%  (최대 43.737%)
  라벨 분포    :
      0: 41,314 (95.18%)
      1: 2,091 (4.82%)

  → 저장: data\processed\polish.parquet  (20.6 MB)


---
## 2. 슬로바키아 SME (Mendeley, DOI 10.17632/j89csb932y.2) — 수동 다운로드

**논문 A·B의 주력 데이터.** 업종 4종(농업·건설·제조·소매)이 명시된 유일한 공개 SME 부도 데이터.

Mendeley는 세션 토큰 때문에 자동 다운로드가 불안정합니다. 아래 절차로 수동 다운로드하세요.

1. https://data.mendeley.com/datasets/j89csb932y/2 접속
2. 전체 파일 다운로드 (zip)
3. 압축을 풀어 내용물을 아래 경로에 배치:
   `data/raw/slovak/`

이 셀은 파일이 있으면 검증·저장하고, 없으면 안내만 출력합니다.

In [7]:
SLOVAK_DIR = RAW / "slovak"
SLOVAK_DIR.mkdir(exist_ok=True)

files = [p for p in SLOVAK_DIR.rglob("*") if p.is_file()
         and p.suffix.lower() in {".csv", ".xlsx", ".xls"}]

if not files:
    print("⚠️ 슬로바키아 데이터가 아직 없습니다.")
    print(f"   → {SLOVAK_DIR} 에 Mendeley 파일을 배치한 뒤 이 셀을 다시 실행하세요.")
    print("   https://data.mendeley.com/datasets/j89csb932y/2")
else:
    print(f"발견된 파일 {len(files)}개:")
    for f in files:
        print("   -", f.relative_to(RAW))
    # 파일 구조는 업종·연도별로 나뉘어 있을 수 있음 → 실제 확인 후 통합 로직 조정
    print("\n※ 다음 노트북(01_eda)에서 업종·연도 구조를 확인하고 통합 스키마를 확정하세요.")


발견된 파일 32개:
   - slovak\bankrupt_agriculture_13_year_10_11_12.csv
   - slovak\bankrupt_agriculture_14_year_11_12_13.csv
   - slovak\bankrupt_agriculture_15_year_12_13_14.csv
   - slovak\bankrupt_agriculture_16_year_13_14_15.csv
   - slovak\bankrupt_construction_13_year_10_11_12.csv
   - slovak\bankrupt_construction_14_year_11_12_13.csv
   - slovak\bankrupt_construction_15_year_12_13_14.csv
   - slovak\bankrupt_construction_16_year_13_14_15.csv
   - slovak\bankrupt_manufacture_13_year_10_11_12.csv
   - slovak\bankrupt_manufacture_14_year_11_12_13.csv
   - slovak\bankrupt_manufacture_15_year_12_13_14.csv
   - slovak\bankrupt_manufacture_16_year_13_14_15.csv
   - slovak\bankrupt_retail_13_year_10_11_12.csv
   - slovak\bankrupt_retail_14_year_11_12_13.csv
   - slovak\bankrupt_retail_15_year_12_13_14.csv
   - slovak\bankrupt_retail_16_year_13_14_15.csv
   - slovak\nonbankrupt_agriculture_13_year_10_11_12.csv
   - slovak\nonbankrupt_agriculture_14_year_11_12_13.csv
   - slovak\nonbankrupt_ag

---
## 3. 중국 SMEsD (GitHub: shaopengw/ComRisk) — git clone

**논문 C의 주력 데이터.** 소송 이벤트 + 기업 지식그래프 + 부도 라벨. 재무비율 없음.

repo에 `data/` 폴더와 코드가 함께 있습니다.

> ⚠️ 원 코드 환경은 **python 3.7.10 / torch 1.8.1 / PyG 1.7.0** 으로 매우 구버전입니다.
> 여기서는 **데이터만 클론**하고, GNN 재현은 논문 C 착수 시 별도 컨테이너에서 진행하세요.

In [8]:
SMESD_DIR = EXTERNAL / "ComRisk"

if SMESD_DIR.exists() and any(SMESD_DIR.iterdir()):
    print("ComRisk 이미 클론됨:", SMESD_DIR.relative_to(ROOT))
else:
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/shaopengw/ComRisk.git", str(SMESD_DIR)],
            check=True, capture_output=True, text=True,
        )
        print("클론 완료:", SMESD_DIR.relative_to(ROOT))
    except subprocess.CalledProcessError as e:
        print("⚠️ git clone 실패:", e.stderr)

if SMESD_DIR.exists():
    print("\n포함 파일:")
    for p in sorted(SMESD_DIR.rglob("*")):
        if p.is_file() and ".git/" not in str(p):
            print(f"   {p.relative_to(SMESD_DIR)}  ({p.stat().st_size/1e3:.0f} KB)")


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\User\anaconda3\envs\credit_override_env\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\User\anaconda3\envs\credit_override_env\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\User\anaconda3\envs\credit_override_env\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'cp949' codec can't decode byte 0xed in position 38: illegal multibyte sequence


클론 완료: data\external\ComRisk

포함 파일:
   .git\config  (0 KB)
   .git\description  (0 KB)
   .git\HEAD  (0 KB)
   .git\hooks\applypatch-msg.sample  (0 KB)
   .git\hooks\commit-msg.sample  (2 KB)
   .git\hooks\fsmonitor-watchman.sample  (5 KB)
   .git\hooks\post-update.sample  (0 KB)
   .git\hooks\pre-applypatch.sample  (0 KB)
   .git\hooks\pre-commit.sample  (2 KB)
   .git\hooks\pre-merge-commit.sample  (0 KB)
   .git\hooks\pre-push.sample  (1 KB)
   .git\hooks\pre-rebase.sample  (5 KB)
   .git\hooks\pre-receive.sample  (1 KB)
   .git\hooks\prepare-commit-msg.sample  (1 KB)
   .git\hooks\push-to-checkout.sample  (3 KB)
   .git\hooks\sendemail-validate.sample  (2 KB)
   .git\hooks\update.sample  (4 KB)
   .git\index  (1 KB)
   .git\info\exclude  (0 KB)
   .git\logs\HEAD  (0 KB)
   .git\logs\refs\heads\main  (0 KB)
   .git\logs\refs\remotes\origin\HEAD  (0 KB)
   .git\objects\pack\pack-3c12c634ecbed9004e8bf30cad5603a98e8acaab.idx  (2 KB)
   .git\objects\pack\pack-3c12c634ecbed9004e8bf30cad

**SMEsD 데이터 형식 메모** (clone 후 확인됨)

`data/` 안은 CSV가 아니라 **pickle(.pkl)** 입니다:
- `train_data.pkl` / `validate_data.pkl` / `test_data.pkl` — 시점 기반 분할 (2014-18 / 2019 / 2020-21)
- `split_data_idx.pkl` — 분할 인덱스
- `node2index.pkl` — 노드 매핑
- `meta_emb.pkl` — metapath2vec 사전학습 임베딩

확인 결과 이 pickle들은 순수 `list`/`dict` 구조라 **최신 환경에서도 정상 로드**됩니다
(node2index: 6,381개 노드, train/valid/test 각 5개 요소 리스트).
다만 원 학습 파이프라인(`train.py`)은 PyG 1.7.0 API에 묶여 있으므로,
GNN **재현 실행**은 논문 C 착수 시 원 환경 컨테이너에서 하세요.
데이터 로드 자체는 여기서 검증합니다.

In [9]:
import pickle

pkl_files = sorted((SMESD_DIR / "data").glob("*.pkl")) if SMESD_DIR.exists() else []
if pkl_files:
    print("SMEsD pickle 구조 확인:\n")
    for pf in pkl_files:
        try:
            with open(pf, "rb") as f:
                obj = pickle.load(f)
            info = f"(len={len(obj)})" if hasattr(obj, "__len__") else ""
            print(f"  ✅ {pf.name:22s} → {type(obj).__name__} {info}")
        except Exception as e:
            print(f"  ⚠️ {pf.name:22s} → {type(e).__name__} — 논문 C 원 환경에서 처리")
    print("\n※ 노드/엣지 상세 구조는 논문 C의 00_load_smesd 노트북에서 파헤칩니다.")
else:
    print("pkl 파일 없음 — SMEsD 클론을 먼저 완료하세요.")


SMEsD pickle 구조 확인:

  ✅ meta_emb.pkl           → list (len=2)
  ✅ node2index.pkl         → dict (len=6381)
  ✅ split_data_idx.pkl     → list (len=3)
  ✅ test_data.pkl          → list (len=5)
  ✅ train_data.pkl         → list (len=5)
  ✅ validate_data.pkl      → list (len=5)

※ 노드/엣지 상세 구조는 논문 C의 00_load_smesd 노트북에서 파헤칩니다.


---
## 4. 텍스트 기반 부도예측 데이터 (Mendeley, stf3kg7fw3) — 수동 다운로드

**논문 D의 주력 텍스트 소스.** 10-K(MD&A·Risk Factors) + 트위터 텍스트 + 수치.

- 텍스트 부분은 Mendeley에서 무료로 그대로 사용 가능 (**WRDS 불요**)
- 수치 부분 출처는 Compustat → 수치 결합이 필요할 때만 WRDS 검토

절차:
1. https://data.mendeley.com/datasets/stf3kg7fw3/1 접속
2. 다운로드 후 압축 해제
3. `data/raw/text_bankruptcy/` 에 배치

In [11]:
TEXT_DIR = RAW / "text_bankruptcy"
TEXT_DIR.mkdir(exist_ok=True)

files = [p for p in TEXT_DIR.rglob("*") if p.is_file()]
if not files:
    print("⚠️ 텍스트 데이터가 아직 없습니다.")
    print(f"   → {TEXT_DIR} 에 Mendeley 파일을 배치한 뒤 재실행하세요.")
    print("   https://data.mendeley.com/datasets/stf3kg7fw3/1")
else:
    print(f"발견된 파일 {len(files)}개:")
    for f in files[:20]:
        print(f"   {f.relative_to(TEXT_DIR)}  ({f.stat().st_size/1e6:.2f} MB)")
    if len(files) > 20:
        print(f"   ... 외 {len(files)-20}개")


발견된 파일 11개:
   39FV_2021June(2).csv  (2.44 MB)
   NUM10K_X_test_2021June.csv  (20.71 MB)
   NUM10K_X_train_2021June.csv  (17.57 MB)
   NUM10K_y_test_2021June.csv  (0.00 MB)
   NUM10K_y_train_2021June.csv  (0.00 MB)
   NUMtw_X_test_2021June.csv  (0.11 MB)
   NUMtw_X_train_2021June.csv  (0.43 MB)
   NUMtw_y_test_2021June.csv  (0.00 MB)
   NUMtw_y_train_2021June.csv  (0.00 MB)
   sic.csv  (5.77 MB)
   slang.txt  (0.00 MB)


---
## 5. Taiwan Company Bankruptcy (Kaggle) — Kaggle API

**범용 보조 데이터.** 95개 재무비율 + 부도, R&D 비율 포함. 논문 B의 업종 일반화 보조 후보.

`kaggle.json` 이 `~/.kaggle/` 에 있어야 합니다 (권한 600).

In [12]:
TAIWAN_DIR = EXTERNAL / "taiwan"
TAIWAN_DIR.mkdir(exist_ok=True)
csvs = list(TAIWAN_DIR.glob("*.csv"))

if csvs:
    print("Taiwan 이미 존재:", [c.name for c in csvs])
else:
    try:
        subprocess.run(
            ["kaggle", "datasets", "download",
             "-d", "fedesoriano/company-bankruptcy-prediction",
             "-p", str(TAIWAN_DIR), "--unzip"],
            check=True, capture_output=True, text=True,
        )
        csvs = list(TAIWAN_DIR.glob("*.csv"))
        print("다운로드 완료:", [c.name for c in csvs])
    except FileNotFoundError:
        print("⚠️ kaggle CLI 미설치. pip install kaggle 후 재시도.")
    except subprocess.CalledProcessError as e:
        print("⚠️ Kaggle 다운로드 실패:", e.stderr)
        print("   ~/.kaggle/kaggle.json 존재 및 권한(chmod 600) 확인하세요.")

if csvs:
    taiwan = pd.read_csv(csvs[0])
    label = "Bankrupt?" if "Bankrupt?" in taiwan.columns else taiwan.columns[0]
    report(taiwan, "Taiwan (Kaggle)", label_col=label)
    save_processed(taiwan, "taiwan.parquet")


다운로드 완료: ['data.csv']
── Taiwan (Kaggle) ──
  shape        : 6,819 rows × 96 cols
  결측 평균    : 0.000%  (최대 0.000%)
  라벨 분포    :
      0: 6,599 (96.77%)
      1: 220 (3.23%)

  → 저장: data\processed\taiwan.parquet  (4.7 MB)


---
## 6. 수집 현황 요약

각 데이터의 확보 여부를 한눈에 확인합니다. 다음 단계(`01_eda`) 진입 전 체크리스트로 사용하세요.

In [13]:
status = {
    "Polish (UCI)":        (PROCESSED / "polish.parquet").exists(),
    "슬로바키아 SME":       any((RAW / "slovak").rglob("*.csv")) or any((RAW / "slovak").rglob("*.xlsx")),
    "중국 SMEsD":          (EXTERNAL / "ComRisk").exists() and any((EXTERNAL / "ComRisk").iterdir()),
    "텍스트셋":            any((RAW / "text_bankruptcy").rglob("*")),
    "Taiwan (Kaggle)":     (PROCESSED / "taiwan.parquet").exists(),
}
print("데이터 수집 현황")
print("=" * 40)
for name, ok in status.items():
    print(f"  {'✅' if ok else '⬜'}  {name}")
print("=" * 40)
done = sum(status.values())
print(f"  {done}/5 확보 완료")
if done < 5:
    print("\n미확보 항목: 위 해당 셀의 안내(수동 다운로드/API 키)를 따라 처리 후 재실행하세요.")


데이터 수집 현황
  ✅  Polish (UCI)
  ✅  슬로바키아 SME
  ✅  중국 SMEsD
  ✅  텍스트셋
  ✅  Taiwan (Kaggle)
  5/5 확보 완료


In [14]:
import pandas as pd
from pathlib import Path

f = Path("data/raw/slovak/bankrupt_agriculture_13_year_10_11_12.csv")
df = pd.read_csv(f)
print("shape:", df.shape)
print("\n컬럼명:")
print(list(df.columns))
print("\n앞 3행:")
print(df.head(3))
print("\n구분자/인코딩 이슈 있으면 알려주세요")

shape: (6, 1)

컬럼명:
[';"V1";"V2";"V3";"V4";"V5";"V6";"V7";"V8";"V9";"V10";"V11";"V12";"V13";"V14";"V15";"V16";"V17";"V18";"V19";"V20";"V21";"V22";"V23";"V24";"V25";"V26";"V27";"V28";"V29";"V30";"V31";"V32";"V33";"V34";"V35";"V36";"V37";"V38";"V39";"V40";"V41";"V42";"V43";"V44";"V45";"V46";"V47";"V48";"V49";"V50";"V51";"V52";"V53";"V54";"V55";"V56";"V57";"V58";"V59";"V60";"V61";"V62";"V63"']

앞 3행:
                                                                                                                                                                                                                                                                                                                                                                                                                                      ;"V1";"V2";"V3";"V4";"V5";"V6";"V7";"V8";"V9";"V10";"V11";"V12";"V13";"V14";"V15";"V16";"V17";"V18";"V19";"V20";"V21";"V22";"V23";"V24";"V25";"V26";"V27";"V28";"V29";"V30";"V31"

In [15]:
import pandas as pd
from pathlib import Path

f = Path("data/raw/slovak/bankrupt_agriculture_13_year_10_11_12.csv")
df = pd.read_csv(
    f,
    sep=";",           # 세미콜론 구분
    decimal=",",       # 쉼표 소수점
    na_values=["NA"],  # NA를 결측으로
    index_col=0,       # 맨 앞 이름없는 인덱스 컬럼
)
print("shape:", df.shape)
print("컬럼:", list(df.columns))
print(df.head(3))
print("\ndtypes 확인 (전부 float/int여야 정상):")
print(df.dtypes.value_counts())

shape: (6, 63)
컬럼: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63']
      V1       V2     V3    V4    V5    V6       V7     V8      V9     V10  \
1   1.40     3.36   6.51  0.03  0.19  0.99  1693.47  60.05  214.55  304.62   
2 -10.20 -1554.84 -34.94  0.05  0.18  0.48  1246.95  94.55  486.57  219.42   
3 -28.39    55.31 -44.27  0.02  0.06  0.58   567.58  27.39  604.53  318.20   

   ...     V54     V55    V56    V57    V58   V59       V60    V61    V62  \
1  ...    1.59    3.32  11.21  69.89   8.25  0.67   7494.34  28.46  53.95   
2  ...   10.39   12.74  18.95  92.15  21.25  0.27  14846.06   4.58  19.96   
3  .

In [17]:
import pandas as pd
from pathlib import Path

f = Path("data/raw/slovak/bankrupt_agriculture_13_year_10_11_12.csv")
df = pd.read_csv(f, sep=";", decimal=",", na_values=["NA"], index_col=0)

# 블록형 가설: V1과 V22와 V43이 같은 지표의 3개 연도라면 상관 높음
print("── 블록형 가설 검정 (V1·V22·V43이 같은 지표?) ──")
print(df[["V1","V22","V43"]].corr().round(2))

print("\n── 인터리브 가설 검정 (V1·V2·V3이 같은 지표?) ──")
print(df[["V1","V2","V3"]].corr().round(2))

# 값 스케일도 힌트: 같은 지표면 범위가 비슷
print("\n── 각 후보의 값 범위 ──")
print(df[["V1","V2","V3","V22","V43"]].describe().round(1).loc[["min","max","mean"]])

── 블록형 가설 검정 (V1·V22·V43이 같은 지표?) ──
       V1   V22   V43
V1   1.00  0.11 -0.08
V22  0.11  1.00 -0.45
V43 -0.08 -0.45  1.00

── 인터리브 가설 검정 (V1·V2·V3이 같은 지표?) ──
      V1    V2    V3
V1  1.00 -0.23  0.74
V2 -0.23  1.00 -0.04
V3  0.74 -0.04  1.00

── 각 후보의 값 범위 ──
        V1      V2    V3   V22   V43
min  -33.7 -1554.8 -69.4 -15.3 -40.2
max    1.4   155.7   6.5   0.8   7.8
mean -14.9  -215.7 -35.5  -4.1  -5.4


In [16]:
f2 = Path("data/raw/slovak/nonbankrupt_manufacture_16_year_13_14_15.csv")
df2 = pd.read_csv(f2, sep=";", decimal=",", na_values=["NA"], index_col=0)
print("nonbankrupt manufacture 표본수:", len(df2))
print("\n블록형(V1·V22·V43):")
print(df2[["V1","V22","V43"]].corr().round(2))
print("\n인터리브(V1·V2·V3):")
print(df2[["V1","V2","V3"]].corr().round(2))

nonbankrupt manufacture 표본수: 5840

블록형(V1·V22·V43):
       V1   V22   V43
V1   1.00 -0.06  0.00
V22 -0.06  1.00  0.01
V43  0.00  0.01  1.00

인터리브(V1·V2·V3):
      V1    V2   V3
V1  1.00  0.05  0.0
V2  0.05  1.00  0.0
V3  0.00  0.00  1.0


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

f2 = Path("data/raw/slovak/nonbankrupt_manufacture_16_year_13_14_15.csv")
df2 = pd.read_csv(f2, sep=";", decimal=",", na_values=["NA"], index_col=0)

# 1) 컬럼별 결측률 — 연도 블록이 통째로 비면 결측률이 계단식으로 나뉜다
miss = df2.isna().mean()
print("── 컬럼별 결측률 (%) ──")
for i in range(0, 63, 7):
    chunk = miss.iloc[i:i+7]
    print(f"V{i+1:>2}-V{i+7:>2}: " + " ".join(f"{v*100:4.0f}" for v in chunk))

# 2) 블록 경계 후보: 결측률이 급변하는 지점
print("\n── 결측률 요약 ──")
print(f"V1-V21  평균결측: {miss.iloc[0:21].mean()*100:.1f}%")
print(f"V22-V42 평균결측: {miss.iloc[21:42].mean()*100:.1f}%")
print(f"V43-V63 평균결측: {miss.iloc[42:63].mean()*100:.1f}%")

# 3) 값 분포로 지표 정체성 확인: V1과 V22가 같은 지표(다른 연도)면 분포가 유사
print("\n── 분포 유사성: 같은 지표면 사분위가 비슷 ──")
for cols in [("V1","V22","V43"), ("V2","V23","V44"), ("V7","V28","V49")]:
    print(f"\n{cols}:")
    print(df2[list(cols)].describe(percentiles=[.25,.5,.75]).round(1).loc[["25%","50%","75%"]])

── 컬럼별 결측률 (%) ──
V 1-V 7:    1    1    6    4    4    4    5
V 8-V14:   10    6   30    4    4    1    1
V15-V21:    1    4   24   32    1    5    2
V22-V28:    1    1   15    1    1    1    2
V29-V35:    7    2   27    1    1    1    1
V36-V42:    1    1   21   80    1    2    2
V43-V49:    1    1   15    1    1    1    2
V50-V56:    6    2   28    1    1    1    1
V57-V63:    1    1   19   93    1    2    1

── 결측률 요약 ──
V1-V21  평균결측: 6.9%
V22-V42 평균결측: 8.2%
V43-V63 평균결측: 8.5%

── 분포 유사성: 같은 지표면 사분위가 비슷 ──

('V1', 'V22', 'V43'):
       V1   V22   V43
25%   0.0   0.3   0.8
50%   2.8   4.6   5.5
75%  11.0  13.6  14.2

('V2', 'V23', 'V44'):
       V2   V23   V44
25%   0.2   1.7   3.0
50%   9.5  15.5  15.7
75%  33.2  36.9  34.3

('V7', 'V28', 'V49'):
        V7    V28    V49
25%  148.2  154.1  155.7
50%  232.2  236.7  236.7
75%  374.2  381.5  380.9


In [20]:
# =============================================================================
# 00 · Data Acquisition
# Corporate credit-rating research portfolio — 5 public datasets
#
# Run top-to-bottom once. Idempotent: already-downloaded files are skipped.
# Raw files in data/raw/ are read-only; processed outputs go to data/processed/.
#
# | Dataset              | Method        | Automated | Papers        |
# |----------------------|---------------|-----------|---------------|
# | Polish (UCI #365)    | ucimlrepo     | full      | A (robustness)|
# | Slovak SME (Mendeley)| manual DL     | semi      | A, B (main)   |
# | China SMEsD (GitHub) | git clone     | full      | C (main), D   |
# | Text dataset (Mend.) | manual DL     | semi      | D (main)      |
# | Taiwan (Kaggle)      | kaggle API    | full      | general aux   |
# =============================================================================

# --- Dependencies (uncomment on first run) -----------------------------------
# %pip install ucimlrepo kaggle pandas pyarrow

import subprocess
import pickle
from pathlib import Path
import pandas as pd

# --- Project scaffolding -----------------------------------------------------
# Resolve project root whether the notebook runs from the repo root or notebooks/
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]

DATA, RAW = ROOT / "data", ROOT / "data" / "raw"
PROCESSED, EXTERNAL = DATA / "processed", DATA / "external"

for p in [RAW, PROCESSED, EXTERNAL, ROOT / "src", ROOT / "configs",
          ROOT / "artifacts" / "models", ROOT / "artifacts" / "figures",
          ROOT / "artifacts" / "tables"]:
    p.mkdir(parents=True, exist_ok=True)


def rel(p: Path) -> str:
    """Return a repo-relative path string, so absolute paths (which may contain
    a username) are never printed. Falls back to the bare name if outside root."""
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


# Keep raw data and large artifacts out of git
gitignore = ROOT / ".gitignore"
rules = ["data/raw/", "data/processed/", "data/external/", "artifacts/models/",
         "*.parquet", "__pycache__/", ".ipynb_checkpoints/", ".env"]
existing = gitignore.read_text().splitlines() if gitignore.exists() else []
new = [r for r in rules if r not in existing]
if new:
    with gitignore.open("a") as f:
        f.write(("\n" if existing else "") + "\n".join(new) + "\n")

print("Project scaffolding ready.\n")


# --- Shared helpers ----------------------------------------------------------
def report(df: pd.DataFrame, name: str, label_col: str | None = None) -> None:
    """Print a compact validation summary: shape, missingness, label balance."""
    print(f"── {name} ──")
    print(f"  shape        : {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    miss = df.isna().mean()
    print(f"  missing (avg): {miss.mean():.3%}  (max {miss.max():.3%})")
    if label_col and label_col in df.columns:
        print("  label balance:")
        for k, v in df[label_col].value_counts(dropna=False).items():
            print(f"      {k}: {v:,} ({v / len(df):.2%})")
    print()


def save_processed(df: pd.DataFrame, filename: str) -> Path:
    """Write a processed dataframe to data/processed/ as parquet."""
    out = PROCESSED / filename
    df.to_parquet(out, index=False)
    print(f"  → saved: {rel(out)}  ({out.stat().st_size / 1e6:.1f} MB)")
    return out


# =============================================================================
# 1. Polish Companies Bankruptcy (UCI #365) — fully automated
#    64 financial ratios + bankruptcy label, 5 forecasting horizons.
#    License: CC BY 4.0. Used only as a robustness set in Paper A.
# =============================================================================
def fetch_polish() -> pd.DataFrame:
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=365)
    df = pd.concat([ds.data.features, ds.data.targets], axis=1)
    return df.rename(columns={ds.data.targets.columns[0]: "bankrupt"})

try:
    polish = fetch_polish()
except Exception as e:
    # On corporate networks an intercepting CA may cause
    # SSL CERTIFICATE_VERIFY_FAILED. Retry with verification disabled.
    # Use only on a trusted network.
    print(f"⚠️ First attempt failed ({type(e).__name__}: {e}); retrying without SSL verify...")
    try:
        import ssl
        ssl._create_default_https_context = ssl._create_unverified_context
        polish = fetch_polish()
        print("   Fallback succeeded (SSL verification bypassed).")
    except Exception as e2:
        polish = None
        print(f"⚠️ Polish failed: {type(e2).__name__}: {e2}")
        print("   Manual option: download the zip from the UCI page into data/raw/polish/")

if polish is not None:
    report(polish, "Polish (UCI #365)", label_col="bankrupt")
    save_processed(polish, "polish.parquet")


# =============================================================================
# 2. Slovak SME bankruptcy (Mendeley, DOI 10.17632/j89csb932y.2) — manual DL
#    Main dataset for Papers A & B. The only public SME dataset with
#    industry labels (agriculture / construction / manufacture / retail).
#
#    Format verified: European CSV (sep=";", decimal=","), NA for missing,
#    a leading unnamed index column, and 63 numeric columns V1–V63.
#    Structure is BLOCK-wise: 21 ratios × 3 years, i.e.
#        V1–V21  = ratios 1..21 for year_1 (oldest)
#        V22–V42 = ratios 1..21 for year_2
#        V43–V63 = ratios 1..21 for year_3 (closest to evaluation)
#    (Confirmed via a 21-column periodic missingness pattern and near-identical
#     per-ratio distributions across the three blocks.)
#
#    Filename convention:
#        {bankrupt|nonbankrupt}_{industry}_{evalYY}_year_{y1_y2_y3}.csv
#
#    Download: https://data.mendeley.com/datasets/j89csb932y/2
#    → unzip the 32 CSVs into  data/raw/slovak/
# =============================================================================
SLOVAK_DIR = RAW / "slovak"
SLOVAK_DIR.mkdir(exist_ok=True)
slovak_files = sorted(SLOVAK_DIR.glob("*.csv"))

if not slovak_files:
    print("⚠️ Slovak data not found.")
    print("   → place the Mendeley CSVs in data/raw/slovak/, then re-run.")
    print("   https://data.mendeley.com/datasets/j89csb932y/2")
else:
    def load_slovak_file(path: Path) -> pd.DataFrame:
        """Read one Slovak CSV and attach parsed metadata from its filename."""
        df = pd.read_csv(path, sep=";", decimal=",", na_values=["NA"], index_col=0)
        # Rename V1..V63 -> ratioNN_yK  (NN in 1..21, K in 1..3)
        rename = {}
        for col in df.columns:
            n = int(col.lstrip("V"))          # 1..63
            ratio = (n - 1) % 21 + 1          # 1..21
            year = (n - 1) // 21 + 1          # 1..3
            rename[col] = f"ratio{ratio:02d}_y{year}"
        df = df.rename(columns=rename)

        # Parse filename: {label}_{industry}_{evalYY}_year_{y1_y2_y3}.csv
        stem = path.stem
        label = 1 if stem.startswith("bankrupt") else 0
        rest = stem.split("_")
        industry = rest[1]                    # agriculture/construction/manufacture/retail
        eval_year = 2000 + int(rest[2])       # 13 -> 2013
        df.insert(0, "bankrupt", label)
        df.insert(1, "industry", industry)
        df.insert(2, "eval_year", eval_year)
        return df.reset_index(drop=True)

    slovak = pd.concat([load_slovak_file(p) for p in slovak_files], ignore_index=True)
    report(slovak, "Slovak SME (combined)", label_col="bankrupt")

    # Sample counts per industry × eval_year — flags the "rare bankruptcy" risk
    print("  bankrupt count by industry × eval_year:")
    pivot = (slovak[slovak.bankrupt == 1]
             .pivot_table(index="industry", columns="eval_year",
                          values="bankrupt", aggfunc="sum", fill_value=0))
    print(pivot.to_string().replace("\n", "\n  "))
    print()
    save_processed(slovak, "slovak.parquet")


# =============================================================================
# 3. China SMEsD (GitHub: shaopengw/ComRisk) — git clone
#    Main dataset for Paper C. Lawsuit events + enterprise knowledge graph +
#    bankruptcy labels; NO financial ratios.
#
#    Data ships as pickled list/dict objects (not CSV), already split by time:
#        train_data.pkl (2014-18) / validate_data.pkl (2019) / test_data.pkl (2020-21)
#        split_data_idx.pkl, node2index.pkl (6,381 nodes), meta_emb.pkl
#    These load fine on modern Python, but the original GNN training pipeline
#    targets PyG 1.7.0 / torch 1.8.1 — reproduce it in a pinned container
#    when Paper C starts. Here we only clone and verify loadability.
# =============================================================================
SMESD_DIR = EXTERNAL / "ComRisk"

if SMESD_DIR.exists() and any(SMESD_DIR.iterdir()):
    print(f"SMEsD already cloned: {rel(SMESD_DIR)}")
else:
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/shaopengw/ComRisk.git", str(SMESD_DIR)],
            check=True, capture_output=True, text=True)
        print(f"Cloned: {rel(SMESD_DIR)}")
    except subprocess.CalledProcessError as e:
        print(f"⚠️ git clone failed: {e.stderr}")

pkl_files = sorted((SMESD_DIR / "data").glob("*.pkl")) if SMESD_DIR.exists() else []
if pkl_files:
    print("  SMEsD pickle structure:")
    for pf in pkl_files:
        try:
            obj = pickle.load(open(pf, "rb"))
            info = f"(len={len(obj)})" if hasattr(obj, "__len__") else ""
            print(f"    ✅ {pf.name:22s} {type(obj).__name__} {info}")
        except Exception as e:
            print(f"    ⚠️ {pf.name:22s} {type(e).__name__} — handle in Paper C container")
print()


# =============================================================================
# 4. Text-based bankruptcy dataset (Mendeley, stf3kg7fw3) — manual DL
#    Main text source for Paper D: 10-K (MD&A / Risk Factors) + Twitter + numerics.
#    Text-only usage needs NO WRDS. Numeric part comes from Compustat, so only
#    consider WRDS if you later fuse numerics.
#
#    Download: https://data.mendeley.com/datasets/stf3kg7fw3/1
#    → unzip into  data/raw/text_bankruptcy/
# =============================================================================
TEXT_DIR = RAW / "text_bankruptcy"
TEXT_DIR.mkdir(exist_ok=True)
text_files = [p for p in TEXT_DIR.rglob("*") if p.is_file()]

if not text_files:
    print("⚠️ Text dataset not found.")
    print("   → place the Mendeley files in data/raw/text_bankruptcy/, then re-run.")
    print("   https://data.mendeley.com/datasets/stf3kg7fw3/1")
else:
    print(f"Text dataset: {len(text_files)} files found")
    for f in sorted(text_files)[:12]:
        print(f"    {rel(f)}  ({f.stat().st_size / 1e6:.2f} MB)")
    # NOTE: verify whether raw 10-K text (MD&A/Risk Factors) ships separately
    # from the NUM* numeric files before starting Paper D.
print()


# =============================================================================
# 5. Taiwan Company Bankruptcy (Kaggle) — kaggle API
#    General-purpose auxiliary set: 95 financial ratios + bankruptcy label.
#    Requires ~/.kaggle/kaggle.json (chmod 600).
# =============================================================================
TAIWAN_DIR = EXTERNAL / "taiwan"
TAIWAN_DIR.mkdir(exist_ok=True)
taiwan_csvs = list(TAIWAN_DIR.glob("*.csv"))

if not taiwan_csvs:
    try:
        subprocess.run(
            ["kaggle", "datasets", "download",
             "-d", "fedesoriano/company-bankruptcy-prediction",
             "-p", str(TAIWAN_DIR), "--unzip"],
            check=True, capture_output=True, text=True)
        taiwan_csvs = list(TAIWAN_DIR.glob("*.csv"))
    except FileNotFoundError:
        print("⚠️ kaggle CLI not installed. Run: pip install kaggle")
    except subprocess.CalledProcessError as e:
        print(f"⚠️ Kaggle download failed: {e.stderr}")
        print("   Check ~/.kaggle/kaggle.json exists and is chmod 600.")

if taiwan_csvs:
    taiwan = pd.read_csv(taiwan_csvs[0])
    label = "Bankrupt?" if "Bankrupt?" in taiwan.columns else taiwan.columns[0]
    report(taiwan, "Taiwan (Kaggle)", label_col=label)
    save_processed(taiwan, "taiwan.parquet")


# =============================================================================
# 6. Acquisition status — checklist before moving on to 01_eda
# =============================================================================
status = {
    "Polish (UCI)":  (PROCESSED / "polish.parquet").exists(),
    "Slovak SME":    (PROCESSED / "slovak.parquet").exists(),
    "China SMEsD":   SMESD_DIR.exists() and any(SMESD_DIR.iterdir()),
    "Text dataset":  bool(text_files),
    "Taiwan":        (PROCESSED / "taiwan.parquet").exists(),
}
print("Acquisition status")
print("=" * 40)
for name, ok in status.items():
    print(f"  {'✅' if ok else '⬜'}  {name}")
print("=" * 40)
print(f"  {sum(status.values())}/5 acquired")

Project scaffolding ready.

── Polish (UCI #365) ──
  shape        : 43,405 rows × 66 cols
  missing (avg): 1.442%  (max 43.737%)
  label balance:
      0: 41,314 (95.18%)
      1: 2,091 (4.82%)

  → saved: data\processed\polish.parquet  (20.6 MB)
── Slovak SME (combined) ──
  shape        : 51,407 rows × 66 cols
  missing (avg): 7.194%  (max 64.896%)
  label balance:
      0: 51,156 (99.51%)
      1: 251 (0.49%)

  bankrupt count by industry × eval_year:
eval_year     2013  2014  2015  2016
  industry                            
  agriculture      6     6     8     8
  construction    25    30    20    14
  manufacture     30    30    26    14
  retail          12    11     7     4

  → saved: data\processed\slovak.parquet  (8.9 MB)
SMEsD already cloned: data\external\ComRisk
  SMEsD pickle structure:
    ✅ meta_emb.pkl           list (len=2)
    ✅ node2index.pkl         dict (len=6381)
    ✅ split_data_idx.pkl     list (len=3)
    ✅ test_data.pkl          list (len=5)
    ✅ train_dat